# Florida — Title XXXVII (insurance band) → `data/florida/ins_codes/*.md`

Florida Statutes **Title XXXVII** is published under the Legislature’s **Online Sunshine** tree **`0600-0699/`** ([Florida Statutes](https://www.leg.state.fl.us/statutes/)). Insurance-related chapters include **624–632**, **634–636**, **641–642**, **648**, and **651** (see **F.S. 624.01** — “Florida Insurance Code”).

Section HTML lives at **`index.cfm?App_mode=Display_Statute&URL=0600-0699/…/Sections/….html`**. Index pages embed many targets as **`URL=0600-0699/…`** strings. This notebook **BFS**-walks each chapter’s **`*ContentsIndex.html`** pages (following **Part** sub-indexes). **Important:** index links often repeat the same page with **`#menu` / `#content`** fragments; those are **normalized** so discovery does not loop forever.

Each section file is **`FS_sec_<chapter>_<section>.md`** (dots in the section suffix become underscores), with body text from **`#statutes`**.

**Config:** **`CHAPTER_START` / `CHAPTER_END`** (inclusive) default **624–651**. Set **`MAX_CHAPTERS`** to cap how many chapter roots are scanned (**0** = no cap). **`MAX_SECTIONS`** caps **downloads** (**0** = all discovered). **`REUSE_DISCOVERED`** skips a repeat crawl when **`_florida_section_rel_paths.txt`** exists in `OUT_DIR`.

**Politeness:** **`REQUEST_DELAY_SEC`** between HTTP requests.

**SSL:** If certificate verification fails, set **`VERIFY_SSL = False`** (less secure) or refresh **`certifi`**.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q httpx beautifulsoup4 certifi


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path

import certifi
import httpx
from bs4 import BeautifulSoup

DISPLAY_BASE = (
    "https://www.leg.state.fl.us/statutes/index.cfm?App_mode=Display_Statute&URL="
)
TITLE_PREFIX = "0600-0699"

OUT_DIR = Path("data") / "florida" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-FL-Title37/1.0 (public Florida Statutes; educational indexing)"
REQUEST_DELAY_SEC = 0.2
TIMEOUT = 60.0

# Inclusive chapter numbers under TITLE_PREFIX (e.g. 624 → …/0624/…).
CHAPTER_START = 624
CHAPTER_END = 651
# 0 = scan every chapter in the range; otherwise stop after this many *successful* chapter roots.
MAX_CHAPTERS = 0

MAX_SECTIONS = 0
SKIP_EXISTING = True

DISCOVERED_REL_FILE = OUT_DIR / "_florida_section_rel_paths.txt"
REUSE_DISCOVERED = True

VERIFY_SSL = True
_verify = certifi.where() if VERIFY_SSL else False

url_param_re = re.compile(rf"URL=({TITLE_PREFIX}/\d{{4}}/[^\s\"'<>]+)", re.I)
section_path_re = re.compile(
    rf"^{TITLE_PREFIX}/(\d{{4}})/Sections/\d{{4}}\.(.+)\.html$", re.I
)


In [3]:
def norm_rel(p: str) -> str:
    """Strip `#fragment` and trailing `/` so index URLs dedupe correctly."""
    return p.split("#", 1)[0].rstrip("/")


def fetch_html(client: httpx.Client, url: str) -> str:
    r = client.get(url, follow_redirects=True)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return r.text


def discover_sections_in_chapter(client: httpx.Client, ch: int) -> set[str]:
    chap4 = f"{ch:04d}"
    prefix = f"{TITLE_PREFIX}/{chap4}/"
    start = norm_rel(f"{prefix}{chap4}ContentsIndex.html")
    visited: set[str] = set()
    in_q: set[str] = {start}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    while q:
        rel = norm_rel(q.popleft())
        in_q.discard(rel)
        if rel in visited:
            continue
        url = DISPLAY_BASE + rel
        try:
            html = fetch_html(client, url)
        except httpx.HTTPStatusError as e:
            if e.response is not None and e.response.status_code == 404:
                visited.add(rel)
                continue
            raise
        visited.add(rel)
        for raw in url_param_re.findall(html):
            p = norm_rel(raw)
            if not p.startswith(prefix):
                continue
            if "/Sections/" in p and p.lower().endswith(".html"):
                sections.add(p)
            elif "ContentsIndex.html" in p and "Sections" not in p:
                if p not in visited and p not in in_q:
                    in_q.add(p)
                    q.append(p)
    return sections


def discover_all_section_paths(client: httpx.Client) -> list[str]:
    all_paths: set[str] = set()
    roots = 0
    for ch in range(CHAPTER_START, CHAPTER_END + 1):
        chap4 = f"{ch:04d}"
        root_url = DISPLAY_BASE + f"{TITLE_PREFIX}/{chap4}/{chap4}ContentsIndex.html"
        try:
            probe = client.get(root_url, follow_redirects=True)
            time.sleep(REQUEST_DELAY_SEC)
            if probe.status_code == 404:
                continue
            probe.raise_for_status()
        except httpx.HTTPError:
            continue
        roots += 1
        all_paths |= discover_sections_in_chapter(client, ch)
        if MAX_CHAPTERS and roots >= MAX_CHAPTERS:
            break

    def sort_key(rel: str) -> tuple:
        m = section_path_re.match(rel)
        if not m:
            return (9999, (9999,))
        chap = int(m.group(1))
        tail = m.group(2)
        nums: list[int] = []
        for part in tail.split("."):
            if part.isdigit():
                nums.append(int(part))
            else:
                nums.append(-1)
        return (chap, tuple(nums))

    return sorted(all_paths, key=sort_key)


def display_id_from_rel(rel: str) -> str:
    m = section_path_re.match(rel)
    if not m:
        raise ValueError(f"unrecognized section rel: {rel!r}")
    chap = int(m.group(1))
    tail = m.group(2)
    return f"{chap}.{tail}"


def rel_to_filename(rel: str) -> str:
    disp = display_id_from_rel(rel)
    chap, _, rest = disp.partition(".")
    safe_rest = rest.replace(".", "_") if rest else ""
    return f"FS_sec_{chap}_{safe_rest}.md" if safe_rest else f"FS_sec_{chap}.md"


def extract_statute_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    box = soup.find(id="statutes")
    if box:
        return box.get_text("\n", strip=True)
    main = soup.find("main") or soup.find("article")
    if main:
        return main.get_text("\n", strip=True)
    body = soup.find("body")
    return body.get_text("\n", strip=True) if body else soup.get_text("\n", strip=True)


def download_florida() -> dict[str, int]:
    if REUSE_DISCOVERED and DISCOVERED_REL_FILE.exists():
        rels = [
            ln.strip()
            for ln in DISCOVERED_REL_FILE.read_text(encoding="utf-8").splitlines()
            if ln.strip() and not ln.strip().startswith("#")
        ]
        print(f"Loaded {len(rels)} section rel paths from {DISCOVERED_REL_FILE.name}")
    else:
        with httpx.Client(
            headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
            timeout=TIMEOUT,
            verify=_verify,
            http2=False,
        ) as client:
            rels = discover_all_section_paths(client)
        DISCOVERED_REL_FILE.write_text("\n".join(rels) + "\n", encoding="utf-8")
        print(f"Discovered {len(rels)} section rel paths → {DISCOVERED_REL_FILE.resolve()}")

    todo = rels if not MAX_SECTIONS else rels[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited export to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        for i, rel in enumerate(todo, 1):
            dest = OUT_DIR / rel_to_filename(rel)
            if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
                skipped += 1
                continue
            official = DISPLAY_BASE + rel
            disp = display_id_from_rel(rel)
            try:
                html = fetch_html(client, official)
                body = extract_statute_text(html)
            except Exception as e:
                print(f"FAIL {disp}: {e}")
                failed += 1
                continue
            title = f"Florida Statutes § {disp}"
            md = (
                f"# {title}\n\n"
                f"**Florida Statutes — Title XXXVII (`{TITLE_PREFIX}`)**\n\n"
                f"**Official source:** {official}\n\n"
                f"**Section:** {disp}\n\n"
                f"---\n\n"
                f"{body}\n"
            )
            dest.write_text(md, encoding="utf-8")
            wrote += 1
            if i % 200 == 0:
                print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_florida()


Discovered 2279 section rel paths → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/florida/ins_codes/_florida_section_rel_paths.txt
… 200/2279 (wrote=200 skipped=0 failed=0)
… 400/2279 (wrote=400 skipped=0 failed=0)
… 600/2279 (wrote=600 skipped=0 failed=0)
… 800/2279 (wrote=800 skipped=0 failed=0)
… 1000/2279 (wrote=1000 skipped=0 failed=0)
… 1200/2279 (wrote=1200 skipped=0 failed=0)
… 1400/2279 (wrote=1400 skipped=0 failed=0)
… 1600/2279 (wrote=1600 skipped=0 failed=0)
… 1800/2279 (wrote=1800 skipped=0 failed=0)
… 2000/2279 (wrote=2000 skipped=0 failed=0)
… 2200/2279 (wrote=2200 skipped=0 failed=0)
Done. wrote=2279 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/florida/ins_codes


{'wrote': 2279, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
